In [ ]:
# Imports
from typing import Final
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from os import getenv
import rasterio as r
from rasterio.crs import CRS
import matplotlib.pyplot as plt

# Constants
PROJECT_DIR: Final[Path] = Path(find_dotenv(".env", 1, 1)).absolute().parent
load_dotenv(PROJECT_DIR.joinpath(".env"))

LOCAL_DIR: Final[Path] = Path(getenv("LOCAL_DIR"))

# Load example file
tiff_fn = next(LOCAL_DIR.glob("data/tiffs/*.tif"))
with r.open(tiff_fn, "r") as src:
    data = src.read()
    transformer = r.transform.AffineTransformer(src.transform)
    print(src.height, src.width)
    tiff_crs = src.read_crs()

In [ ]:
# visualize
plt.imshow(data[0], "gray_r")

In [ ]:
plt.imshow(data[0, -600: -300, -300:], "grey_r")

### Will use 400 pixel overlap

In [ ]:
from json import load as load_json
with open(PROJECT_DIR.joinpath("config/tiff_to_png.json"), "r") as f:
    png_meta = load_json(f)
png_meta

In [ ]:
def get_partition_img_split_idxs(
    img_dim: int, partsize_dim: int, overlap: int, start: int = 0
) -> list[int]:
    split_idxs = [*range(start, img_dim, partsize_dim - overlap)]

    if (img_dim - split_idxs[-1]) <= overlap:
        # End of image can be fully captured by penultimate partition
        # so drop final split index
        split_idxs = split_idxs[: -1]
    
    # Adjust final split to make the remainder of img_dim equal to
    # partsize dim
    split_idxs[-1] = img_dim - partsize_dim
    return split_idxs

pixel_row_idxs = get_partition_img_split_idxs(
    data.shape[-2], png_meta["png_h"], png_meta["overlap"]
)
pixel_col_idxs = get_partition_img_split_idxs(
    data.shape[-1], png_meta["png_w"], png_meta["overlap"]
)

pixel_rowcol_idxs =\
    [(row, col) for row in pixel_row_idxs for col in pixel_col_idxs]

In [ ]:
from shapely import Point

images, bounds = [], []
for idx, (row, col) in enumerate(pixel_rowcol_idxs, start = 1):
    bounds += [
        {
            "tiff_fn": tiff_fn.name,
            "png_fn": f"{tiff_fn.stem}-{idx}.png",
            "pixel_x": i * (png_meta["png_w"] - 1),
            "pixel_y": j * (png_meta["png_h"] - 1),
            "geometry": Point(transformer.xy(x, y))
        }
        for i, x in enumerate([col, col + png_meta["png_w"] - 1])
        for j, y in enumerate([row, row + png_meta["png_h"] - 1])
    ]
    images.append(data[
        :, row: row + png_meta["png_h"], col: col + png_meta["png_w"]
    ].copy())

In [ ]:
fig, axes = plt\
    .subplots(len(pixel_row_idxs), len(pixel_col_idxs), figsize = (20, 20))
for ax, img in zip(axes.flatten(), images):
    ax.imshow(img[0], cmap = "grey_r")

In [ ]:
from PIL import Image
Image.fromarray((-img[0] + 1) * 255, "L")

In [ ]:
from geopandas import GeoDataFrame
gdf = GeoDataFrame.from_records(bounds)
gdf = gdf.set_crs(tiff_crs)
gdf.crs

In [ ]:
gdf

In [ ]:
tiff_crs.to_wkt()

In [ ]:
from edina import EDINATiffPNGConverter

converter = EDINATiffPNGConverter()
tiff_fn = next(LOCAL_DIR.glob("data/tiffs/*.tif"))
gdf = converter.convert_tiff_to_pngs(
    tiff_path = tiff_fn,
    png_dest = LOCAL_DIR.joinpath("outputs/test_pngs"),
    png_h = png_meta["png_h"],
    png_w = png_meta["png_w"],
    overlap = png_meta["overlap"]
)
gdf